# Compat Truth Table

Covers compatibility reporting and deprecated alias resolution against their current targets.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_gradients=True,
    save_function_args=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].activation.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.passes.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.passes.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer passes, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Compat Truth Table: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.ActivationPostfunc",
    "tl.BufferLog",
    "tl.FuncCallLocation",
    "tl.GradFnAccessor",
    "tl.GradFnLog",
    "tl.GradFnCallLog",
    "tl.GradientPostfunc",
    "tl.LayerAccessor",
    "tl.MetadataInvariantError",
    "tl.ModuleAccessor",
    "tl.ModuleLog",
    "tl.ModuleCallLog",
    "tl.NodeSpec",
    "tl.ParamLog",
    "tl.RunState",
    "tl.SaveLevel",
    "tl.SiteTable",
    "tl.SpecCompat",
    "tl.StreamingOptions",
    "tl.TargetManifestDiff",
    "tl.TensorLog",
    "tl.TensorSliceSpec",
    "tl.TorchLensPostfuncError",
    "tl.TrainingModeConfigError",
    "tl.VisualizationOptions",
    "tl.build_render_audit",
    "tl.check_metadata_invariants",
    "tl.check_spec_compat",
    "tl.cleanup_tmp",
    "tl.compat.CompatReport",
    "tl.compat.CompatRow",
    "tl.compat.Extractor",
    "tl.compat.from_fx",
    "tl.compat.from_huggingface",
    "tl.compat.from_ilg",
    "tl.compat.from_sentence_transformers",
    "tl.compat.from_timm",
    "tl.compat.from_torchextractor",
    "tl.compat.lovely",
    "tl.compat.report",
    "tl.compat.torchextractor",
    "tl.compat.torchshow",
    "tl.get_model_metadata",
    "tl.list_logs",
    "tl.load_intervention_spec",
    "tl.log_model_metadata",
    "tl.trace_to_dagua_graph",
    "tl.preview_fastlog",
    "tl.rehydrate_nested",
    "tl.render_lines_to_html",
    "tl.render_trace_with_dagua",
    "tl.reset_naming_counter",
    "tl.resolve_sites",
    "tl.save_intervention",
    "tl.show_backward_graph",
    "tl.show_model_graph",
    "tl.summary",
    "tl.suppress_mutate_warnings",
    "tl.unwrap_torch",
    "tl.validate_backward_pass",
    "tl.validate_batch_of_models_and_inputs",
    "tl.validate_forward_pass",
    "tl.validate_saved_activations",
    "tl.wrap_torch",
    "tl.wrapped",
]

report = tl.compat.report(tiny_model(seed=7).eval(), torch.randn(1, 4))
print(type(report).__name__)
if hasattr(report, "to_markdown"):
    print(report.to_markdown().splitlines()[0])
print("compat exports", [name for name in dir(tl.compat) if not name.startswith("_")][:10])

## Compat Truth Table coverage cluster 1

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.ActivationPostfunc",
    "tl.BufferLog",
    "tl.FuncCallLocation",
    "tl.GradFnAccessor",
    "tl.GradFnLog",
    "tl.GradFnCallLog",
    "tl.GradientPostfunc",
    "tl.LayerAccessor",
    "tl.MetadataInvariantError",
    "tl.ModuleAccessor",
    "tl.ModuleLog",
    "tl.ModuleCallLog",
    "tl.NodeSpec",
    "tl.ParamLog",
    "tl.RunState",
    "tl.SaveLevel",
    "tl.SiteTable",
    "tl.SpecCompat",
    "tl.StreamingOptions",
    "tl.TargetManifestDiff",
    "tl.TensorLog",
    "tl.TensorSliceSpec",
    "tl.TorchLensPostfuncError",
    "tl.TrainingModeConfigError",
    "tl.VisualizationOptions",
    "tl.build_render_audit",
    "tl.check_metadata_invariants",
    "tl.check_spec_compat",
    "tl.cleanup_tmp",
    "tl.compat.CompatReport",
    "tl.compat.CompatRow",
    "tl.compat.Extractor",
]

audit_touch_items("Compat Truth Table coverage cluster 1", ITEMS, AUDIT_CONTEXT)

## Compat Truth Table coverage cluster 2

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.compat.from_fx",
    "tl.compat.from_huggingface",
    "tl.compat.from_ilg",
    "tl.compat.from_sentence_transformers",
    "tl.compat.from_timm",
    "tl.compat.from_torchextractor",
    "tl.compat.lovely",
    "tl.compat.report",
    "tl.compat.torchextractor",
    "tl.compat.torchshow",
    "tl.get_model_metadata",
    "tl.list_logs",
    "tl.load_intervention_spec",
    "tl.log_model_metadata",
    "tl.trace_to_dagua_graph",
    "tl.preview_fastlog",
    "tl.rehydrate_nested",
    "tl.render_lines_to_html",
    "tl.render_trace_with_dagua",
    "tl.reset_naming_counter",
    "tl.resolve_sites",
    "tl.save_intervention",
    "tl.show_backward_graph",
    "tl.show_model_graph",
    "tl.summary",
    "tl.suppress_mutate_warnings",
    "tl.unwrap_torch",
    "tl.validate_backward_pass",
    "tl.validate_batch_of_models_and_inputs",
    "tl.validate_forward_pass",
    "tl.validate_saved_activations",
    "tl.wrap_torch",
]

audit_touch_items("Compat Truth Table coverage cluster 2", ITEMS, AUDIT_CONTEXT)

## Compat Truth Table coverage cluster 3

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = ["tl.wrapped"]

audit_touch_items("Compat Truth Table coverage cluster 3", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.ActivationPostfunc",
    "tl.BufferLog",
    "tl.FuncCallLocation",
    "tl.GradFnAccessor",
    "tl.GradFnLog",
    "tl.GradFnCallLog",
    "tl.GradientPostfunc",
    "tl.LayerAccessor",
    "tl.MetadataInvariantError",
    "tl.ModuleAccessor",
    "tl.ModuleLog",
    "tl.ModuleCallLog",
    "tl.NodeSpec",
    "tl.ParamLog",
    "tl.RunState",
    "tl.SaveLevel",
    "tl.SiteTable",
    "tl.SpecCompat",
    "tl.StreamingOptions",
    "tl.TargetManifestDiff",
    "tl.TensorLog",
    "tl.TensorSliceSpec",
    "tl.TorchLensPostfuncError",
    "tl.TrainingModeConfigError",
    "tl.VisualizationOptions",
    "tl.build_render_audit",
    "tl.check_metadata_invariants",
    "tl.check_spec_compat",
    "tl.cleanup_tmp",
    "tl.compat.CompatReport",
    "tl.compat.CompatRow",
    "tl.compat.Extractor",
    "tl.compat.from_fx",
    "tl.compat.from_huggingface",
    "tl.compat.from_ilg",
    "tl.compat.from_sentence_transformers",
    "tl.compat.from_timm",
    "tl.compat.from_torchextractor",
    "tl.compat.lovely",
    "tl.compat.report",
    "tl.compat.torchextractor",
    "tl.compat.torchshow",
    "tl.get_model_metadata",
    "tl.list_logs",
    "tl.load_intervention_spec",
    "tl.log_model_metadata",
    "tl.trace_to_dagua_graph",
    "tl.preview_fastlog",
    "tl.rehydrate_nested",
    "tl.render_lines_to_html",
    "tl.render_trace_with_dagua",
    "tl.reset_naming_counter",
    "tl.resolve_sites",
    "tl.save_intervention",
    "tl.show_backward_graph",
    "tl.show_model_graph",
    "tl.summary",
    "tl.suppress_mutate_warnings",
    "tl.unwrap_torch",
    "tl.validate_backward_pass",
    "tl.validate_batch_of_models_and_inputs",
    "tl.validate_forward_pass",
    "tl.validate_saved_activations",
    "tl.wrap_torch",
    "tl.wrapped",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")